# Lesson 11 — Training Loop, Dataset & DataLoader

## 学习目标

前面的课程已经分别解决了神经网络训练中的几个核心组件：

- **Embedding**：把离散的 Token ID 映射成连续向量；
- **Forward Pass**：根据当前参数计算模型输出；
- **Cross Entropy**：衡量预测分布与正确 Token 之间的差异；
- **Autograd**：通过 `loss.backward()` 计算梯度；
- **Optimizer**：根据梯度更新参数。

但是，到目前为止这些模块基本还是独立存在的。

这一节要第一次把它们连接成真正的训练过程：

`Dataset → DataLoader → Mini-Batch → Forward → Loss → Backward → Optimizer Step`

完成本节后，应能够：

1. 理解 `Dataset`、`DataLoader` 和 Mini-Batch 各自解决什么问题；
2. 理解训练数据为什么通常需要 Batch Dimension；
3. 自己构造一个最小 `Dataset`；
4. 使用 `DataLoader` 完成 batching 和 shuffling；
5. 写出标准 PyTorch Training Loop；
6. 理解 `zero_grad()` 为什么必须存在；
7. 理解 `model.train()` 与 `model.eval()`；
8. 编写 Validation Loop；
9. 理解 Gradient Norm；
10. 使用 Gradient Clipping；
11. 记录 loss、learning rate 等训练指标；
12. 保存和恢复 checkpoint；
13. 最终把前面的知识组合成一个可以真正学习参数的神经网络。

这一节最重要的目标不是记住某个 PyTorch 模板，而是理解：

> **一次训练迭代中，Tensor、Gradient 和 Parameter 到底经历了什么。**


## 1. 从 Optimization 到 Training

上一节研究的是单次参数更新。

假设模型参数为 $\theta$，当前 mini-batch 为 $(x,y)$，模型首先计算预测：

$$
\hat{y}=f_\theta(x)
$$

随后计算 loss，例如：

$$
L=\operatorname{CrossEntropy}(\hat{y},y)
$$

Autograd 计算梯度 $\nabla_\theta L$，Optimizer 再利用这个梯度更新参数。

以最简单的 SGD 为例：

$$
\theta_{t+1}
=
\theta_t-\eta\nabla_\theta L
$$

其中 $\eta$ 是 learning rate。

但真实训练不会只进行一次更新，而是不断重复：

1. 从数据集中取得一个 mini-batch；
2. Forward；
3. 计算 Loss；
4. Backward；
5. 更新 Parameter；
6. 继续下一个 mini-batch。

因此 Training Loop 本质上是：

`Data → Compute → Gradient → Update → Repeat`

后面无论训练 Tiny MLP、Transformer，还是完整 Language Model，这个基本结构都不会改变。


## 2. Dataset 是什么？

训练模型之前，首先需要定义：

> **一个训练样本到底是什么？**

假设现在做一个非常简单的回归问题：

`input x → target y`

例如数据满足近似关系：

$y = 3x + 2$

那么一个 sample 可以表示为：

`(x_i, y_i)`

整个 Dataset 则是很多 sample 的集合：

$$
\mathcal{D}
=
\{(x_1,y_1),(x_2,y_2),\ldots,(x_N,y_N)\}
$$

PyTorch 中，一个最基本的 map-style `Dataset` 通常需要实现两个操作：

- `__len__()`：数据集中有多少个 sample；
- `__getitem__(index)`：给定 index，返回对应 sample。

因此可以把 Dataset 理解为：

> **定义“如何根据索引得到一个训练样本”的对象。**

它本身并不负责梯度下降，也不负责更新模型参数。


In [1]:
from __future__ import annotations

import torch
from torch import Tensor
from torch.utils.data import Dataset


class LinearDataset(Dataset):
    def __init__(self, x: Tensor, y: Tensor) -> None:
        if x.shape[0] != y.shape[0]:
            raise ValueError("x and y must have the same length")

        self.x = x
        self.y = y

    def __len__(self) -> int:

        return self.x.shape[0]

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor]:

        return self.x[index], self.y[index]


x = torch.arange(10, dtype=torch.float32).unsqueeze(-1)
y = 3.0 * x + 2.0

dataset = LinearDataset(x=x, y=y)

print("number of samples:", len(dataset))
print("sample 0:", dataset[0])
print("sample 5:", dataset[5])

number of samples: 10
sample 0: (tensor([0.]), tensor([2.]))
sample 5: (tensor([5.]), tensor([17.]))


### Dataset 中的 Shape Thinking

这里：

- `x.shape == (10, 1)`
- `y.shape == (10, 1)`

第一维的 `10` 表示 sample 数量。

执行：

`dataset[5]`

相当于沿第 0 维取一个样本。

因为整数 indexing 会消除对应维度，所以：

`x[5].shape == (1,)`

注意这里还没有 Batch Dimension。

单个 sample 的 shape 是 `(1,)`；之后经过 DataLoader batching，才会得到例如 `(4,1)`。

这正好对应之前学过的 indexing 规则：

> **整数索引会消除被索引的维度。**


## 3. Mini-Batch

假设 Dataset 中共有 $N$ 个样本。

一种极端方法是每次只使用一个样本更新参数，这通常称为 **Stochastic Gradient Descent** 的基本形式。

另一种极端方法是一次使用整个 Dataset：

$$
L(\theta)
=
\frac{1}{N}
\sum_{i=1}^{N}
L_i(\theta)
$$

然后根据整个 Dataset 的平均梯度进行一次更新。

实际深度学习训练通常采用二者之间的方案：

> **Mini-Batch Training**

假设 batch size 为 $B$，每一步只取 $B$ 个样本。

例如：

`Dataset: 10000 samples`

设置：

`batch_size = 32`

那么一次 Forward 接收到的不是一个 sample，而是一组 sample：

`(32, feature_dim)`

Mini-Batch 有几个重要作用：

1. 可以利用 GPU 的并行计算能力；
2. 不需要一次把整个 Dataset 放进显存；
3. 相比单样本梯度，梯度估计通常更加稳定；
4. 可以在计算效率和梯度噪声之间取得平衡。

后面的 Transformer 中，$B$ 几乎总是表示 **Batch Size**。


### Batch Dimension

假设单个样本：

`x.shape == (D,)`

其中 $D$ 是 feature dimension。

如果一次取 $B$ 个样本，那么 batching 后：

`x_batch.shape == (B, D)`

对于 Language Model，单条 token sequence 通常是：

`tokens.shape == (T,)`

其中 $T$ 是 sequence length。

经过 batching：

`tokens.shape == (B, T)`

Embedding 后：

`hidden.shape == (B, T, D)`

因此前面反复出现的 Transformer Shape：

`(B, T, D)`

现在可以获得更加具体的解释：

| Dimension | Meaning |
|---|---|
| `B` | 当前 mini-batch 中有多少条 sequence |
| `T` | 每条 sequence 有多少个 token |
| `D` | 每个 token 的 hidden dimension |

所以 Batch Dimension 并不是 Transformer 特有的概念。

它来自训练系统最基本的 Mini-Batch 机制。


## 4. DataLoader 做什么？

`Dataset` 解决的是：

> 给定一个 index，怎样获得一个 sample？

但是训练时我们通常还需要：

- 自动组成 mini-batch；
- shuffle 数据；
- 遍历整个 Dataset；
- 后续可能使用多个 worker 加载数据。

这些工作由 `DataLoader` 负责。

最简单的关系可以理解成：

`Dataset → individual samples`

而：

`DataLoader → batches of samples`

例如设置：

`batch_size = 4`

那么 DataLoader 会把四个 sample 组合起来。

如果单个 sample 的 input shape 是 `(1,)`，那么 batch 后通常得到：

`(4, 1)`

也就是说，DataLoader 自动创建了最前面的 Batch Dimension。


In [2]:
from torch.utils.data import DataLoader

loder = DataLoader(dataset, batch_size=4, shuffle=False)

for batch_index, (x_batch, y_batch) in enumerate(loder):
    print(
        f"Batch {batch_index}: x_batch shape = {x_batch.shape}, y_batch shape = {y_batch.shape}"
    )

Batch 0: x_batch shape = torch.Size([4, 1]), y_batch shape = torch.Size([4, 1])
Batch 1: x_batch shape = torch.Size([4, 1]), y_batch shape = torch.Size([4, 1])
Batch 2: x_batch shape = torch.Size([2, 1]), y_batch shape = torch.Size([2, 1])


### 为什么最后一个 Batch 只有 2 个 Sample？

Dataset 一共有 10 个 sample，而：

`batch_size = 4`

因此可以分成：

`4 + 4 + 2 = 10`

所以最后一个 batch 的 shape 是 `(2,1)`。

这说明一个非常重要的事实：

> **默认情况下，不要假设每个 batch 的 `B` 都一定等于 `batch_size`。**

最后一个 batch 可能更小。

如果某些任务必须保证固定 batch size，可以使用：

`drop_last=True`

这样最后不足一个完整 batch 的数据会被丢弃。

在后面的 Transformer 代码中，我们应该尽量根据 Tensor 的真实 shape 工作，而不是把 `B` 写死。


In [3]:
loader_drop_last = DataLoader(dataset, batch_size=4, shuffle=False, drop_last=True)

for x_batch, y_batch in loader_drop_last:
    print(tuple(x_batch.shape), tuple(y_batch.shape))


(4, 1) (4, 1)
(4, 1) (4, 1)


## 5. 为什么训练数据通常需要 Shuffle？

假设 Dataset 按某种规律排序。

例如：

`class 0 → class 0 → ... → class 1 → class 1 → ...`

如果严格按照这个顺序训练，那么连续多个 mini-batch 的数据分布可能非常相似。

这会使每一步得到的 gradient 强烈受到数据排列顺序影响。

训练时通常使用：

`shuffle=True`

让每个 epoch 开始时重新随机排列 sample。

需要注意：

> **Shuffle 改变的是样本被访问的顺序，而不是 Dataset 本身的数学内容。**

Validation / Test 通常不需要 shuffle，因为这些阶段不进行参数更新。


In [4]:
shuffle_loader = DataLoader(dataset, batch_size=4, shuffle=True)

for x_batch, _ in shuffle_loader:
    print(x_batch.squeeze(-1))


tensor([4., 8., 7., 3.])
tensor([2., 6., 1., 5.])
tensor([0., 9.])


## 6. Epoch、Batch 与 Optimization Step

这三个术语以后会不断出现，因此必须区分。

### Epoch

模型完整遍历一次 Training Dataset，称为一个 **epoch**。

如果 Dataset 有 $N$ 个 sample，batch size 为 $B$，那么一个 epoch 大约包含：

$$
\left\lceil \frac{N}{B} \right\rceil
$$

个 batch。

### Batch

一次送入模型的一组训练样本。

例如：

`x_batch.shape == (32, D)`

表示当前 batch 有 32 个 sample。

### Optimization Step

执行一次：

`optimizer.step()`

意味着根据当前 gradient 更新一次 Parameter。

在最普通的 Training Loop 中：

> **一个 batch 通常对应一个 optimization step。**

因此典型结构是：

`Epoch → many Batches → many Optimizer Steps`

例如 Dataset 有 10,000 个 sample，`batch_size=100`，且不考虑 `drop_last`，那么一个 epoch 包含 100 个 batch，也通常对应 100 次 parameter update。


In [5]:
num_samples = 10_000
batch_size = 100

steps_per_epoch = (num_samples + batch_size - 1) // batch_size

print("steps per epoch:", steps_per_epoch)


steps per epoch: 100


## 7. Training Loop 的核心结构

现在可以把前面所有知识第一次连接起来。

标准训练步骤是：

1. 从 DataLoader 取得一个 mini-batch；
2. 清空旧 gradient；
3. Forward；
4. 计算 loss；
5. Backward；
6. Optimizer Step。

对应最核心的 PyTorch 结构：

```python
for x_batch, y_batch in loader:
    optimizer.zero_grad()

    prediction = model(x_batch)
    loss = loss_fn(prediction, y_batch)

    loss.backward()
    optimizer.step()
```

### 一次 Training Step 中发生了什么？

假设当前模型参数记作 $\theta_t$。

首先执行 Forward：

`prediction = model(x_batch)`

得到当前参数下的预测 $f_{\theta_t}(x)$。

随后计算 loss：

`loss = loss_fn(prediction, y_batch)`

得到标量 $L(\theta_t)$。

执行：

`loss.backward()`

Autograd 根据 Computational Graph 计算 $\nabla_\theta L$，并把结果累积到各个 Parameter 的 `.grad` 中。

最后执行：

`optimizer.step()`

Optimizer 读取这些 gradient，并把参数从 $\theta_t$ 更新为 $\theta_{t+1}$。

因此一次 Training Step 可以概括成：

$$
\theta_t
\xrightarrow{\text{forward}}
L(\theta_t)
\xrightarrow{\text{backward}}
\nabla_\theta L
\xrightarrow{\text{optimizer}}
\theta_{t+1}
$$

注意这里使用 display math 是有意义的，因为它描述的是整节最重要的一条状态转换链。


In [7]:
from torch import nn

torch.manual_seed(42)

model = nn.Linear(in_features=1, out_features=1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

loss_fn = nn.MSELoss()

train_loader = DataLoader(dataset, batch_size=4, shuffle=True)

num_epochs = 100

for epoch in range(num_epochs):
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        prediction = model(x_batch)

        loss = loss_fn(prediction, y_batch)
        loss.backward()

        optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"epoch={epoch + 1:03d}", f"loss={loss.item():.6f}")

print("learned weight:", model.weight.item())
print("learned bias:", model.bias.item())

epoch=010 loss=0.037127
epoch=020 loss=0.116587
epoch=030 loss=0.079371
epoch=040 loss=0.016513
epoch=050 loss=0.026493
epoch=060 loss=0.020187
epoch=070 loss=0.020343
epoch=080 loss=0.007341
epoch=090 loss=0.013300
epoch=100 loss=0.007009
learned weight: 3.024386405944824
learned bias: 1.8644695281982422


### 当前模型到底在学习什么？

我们生成的数据满足：

$y = 3x + 2$

而 `nn.Linear(1, 1)` 的模型形式是：

$y = wx + b$

所以理论上的最优参数应该接近：

- $w = 3$
- $b = 2$

训练开始时，`weight` 和 `bias` 是随机初始化的。

Training Loop 的任务就是通过数据不断调整它们，使：

`model(x)`

逐渐接近：

`3 * x + 2`

这个实验虽然非常简单，但它包含了之后训练 Transformer 时完全相同的核心机制：

`Forward → Loss → Backward → Update`


## 9. 不要因为开始训练就停止 Shape Thinking

假设：

`batch_size = 4`

当前：

`x_batch.shape == (4, 1)`

模型是：

`nn.Linear(1, 1)`

那么 Forward：

`prediction = model(x_batch)`

得到：

`prediction.shape == (4, 1)`

Target 同样满足：

`y_batch.shape == (4, 1)`

因此 MSE 可以逐元素比较 prediction 与 target。

完整 shape flow：

`x_batch: (B,1)`

↓

`Linear`

↓

`prediction: (B,1)`

↓

`compare with y_batch: (B,1)`

↓

`MSE reduction`

↓

`loss: scalar`

这里最关键的是：

> **训练代码开始变复杂以后，仍然要持续追踪每一个重要 Tensor 的 shape。**

到了 Transformer，这个习惯会变成：

`tokens: (B,T) → hidden: (B,T,D) → logits: (B,T,V) → loss: scalar`


In [8]:
x_batch, y_batch = next(iter(train_loader))
prediction = model(x_batch)

loss = loss_fn(prediction, y_batch)

print("x_batch:", x_batch.shape)
print("y_batch:", y_batch.shape)
print("prediction:", prediction.shape)
print("loss:", loss.shape)

x_batch: torch.Size([4, 1])
y_batch: torch.Size([4, 1])
prediction: torch.Size([4, 1])
loss: torch.Size([])


## 10. Gradient Accumulation 再次出现

PyTorch 默认会把新的 gradient **累积**到 Parameter 已有的 `.grad` 中。

也就是说，连续执行两次：

`loss.backward()`

并不会自动覆盖第一次 gradient。

如果第一次得到梯度 $g_1$，第二次得到 $g_2$，而中间没有清空，那么 `.grad` 中最终可能保存：

$g_1 + g_2$

因此普通 Training Loop 通常需要在每次新的 optimization step 前执行：

`optimizer.zero_grad()`

典型顺序：

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()


In [9]:
parameter = torch.tensor(2.0, requires_grad=True)

loss_1 = parameter**2
loss_1.backward()

print("after backward 1:", parameter.grad)

loss_2 = parameter**2
loss_2.backward()

print("after backward 2:", parameter.grad)

after backward 1: tensor(4.)
after backward 2: tensor(8.)


In [ ]:
parameter.grad = None

loss_3 = parameter**2
loss_3.backward()

print("after clearing grad:", parameter.grad)

after clearing grad: tensor(4.)


## 11. 清空 Gradient 的现代写法

PyTorch 中常见：

`optimizer.zero_grad()`

当前 PyTorch 的 `Optimizer.zero_grad()` 默认使用：

`set_to_none=True`

因此 gradient 通常会被设置为 `None`，而不是显式写成全零 Tensor。

概念上我们仍然可以把它理解成：

> **清除上一轮 optimization step 留下的 gradient。**

但是在 Debug 时需要知道：

```python
parameter.grad
```
可能是 `None`，也可能是全零 Tensor.

## 12. Training Mode 与 Evaluation Mode

PyTorch 模型通常有两种运行模式：

- Training Mode
- Evaluation Mode

切换方式：

`model.train()`

`model.eval()`

需要特别注意：

> `model.eval()` 并不表示关闭 gradient。

这两个方法主要控制某些 Layer 在训练和推理阶段的不同行为，例如：

- `Dropout`
- `BatchNorm`

Validation 时通常同时使用：

`model.eval()`

以及：

`torch.no_grad()`

二者作用不同：

| Operation | 作用 |
|---|---|
| `model.train()` | 设置 Layer 为 Training Mode |
| `model.eval()` | 设置 Layer 为 Evaluation Mode |
| `torch.no_grad()` | 不构建 gradient graph |

In [11]:
dropout = nn.Dropout(p=0.5)
x_demo = torch.ones(8)

dropout.train()
print("training:", dropout(x_demo))

dropout.eval()
print("evaluation:", dropout(x_demo))

training: tensor([2., 0., 0., 0., 0., 2., 2., 0.])
evaluation: tensor([1., 1., 1., 1., 1., 1., 1., 1.])


## 13. 为什么需要 Validation Set？

如果只观察 Training Loss，只能知道模型是否越来越擅长拟合训练数据。

Validation Set 用来回答另一个问题：

> 模型是否学到了可以推广到未参与参数更新的数据上的规律？

通常数据分为：

- Training Set
- Validation Set
- Test Set

可以简单理解为：

Training Data → Learn

Validation Data → Measure

如果 Training Loss 持续下降，但 Validation Loss 开始上升，常见原因之一是 **Overfitting**。

In [12]:
class LinearDataset(Dataset):
    def __init__(self, x: torch.Tensor, y: torch.Tensor) -> None:
        self.x = x
        self.y = y

    def __len__(self) -> int:
        return self.x.shape[0]

    def __getitem__(self, index: int):
        return self.x[index], self.y[index]


torch.manual_seed(42)

x_all = torch.linspace(-5.0, 5.0, steps=120).unsqueeze(-1)
noise = 0.2 * torch.randn_like(x_all)
y_all = 3.0 * x_all + 2.0 + noise

x_train = x_all[:100]
y_train = y_all[:100]

x_valid = x_all[100:]
y_valid = y_all[100:]

train_dataset = LinearDataset(x_train, y_train)
valid_dataset = LinearDataset(x_valid, y_valid)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False)


## 14. Validation Loop

Training Loop 通常包括：

1. Forward
2. Loss
3. Backward
4. Optimizer Step

Validation Loop 只需要：

1. Forward
2. Loss
3. Metric

Validation 中不应该出现：

`loss.backward()`

`optimizer.step()`

标准结构：

`model.eval()`

然后在 `torch.no_grad()` 中执行 Forward 和 Loss 计算。

In [13]:
model = nn.Linear(in_features=1, out_features=1)
loss_fn = nn.MSELoss()

model.eval()

validation_loss_sum = 0.0
validation_sample_count = 0

with torch.no_grad():
    for x_batch, y_batch in valid_loader:
        prediction = model(x_batch)
        loss = loss_fn(prediction, y_batch)

        batch_size = x_batch.shape[0]

        validation_loss_sum += loss.item() * batch_size
        validation_sample_count += batch_size

validation_loss = validation_loss_sum / validation_sample_count

print("validation loss:", validation_loss)


validation loss: 368.71021728515626


## 15. 正确聚合 Validation Loss

假设 Validation DataLoader 产生三个 batch：

- Batch 1：16 samples
- Batch 2：16 samples
- Batch 3：8 samples

如果 Loss 是每个 batch 内样本的平均值，那么不能简单计算三个 batch loss 的算术平均。

更稳妥的方法是：

$$
L_{\mathrm{valid}}
=
\frac{
\sum_k B_k L_k
}{
\sum_k B_k
}
$$

其中：

- \(B_k\)：第 \(k\) 个 batch 的实际 batch size
- \(L_k\)：该 batch 的平均 loss

这样即使最后一个 batch 更小，也能得到正确的全数据平均 loss。

## 16. Training + Validation 完整结构

一个 epoch 可以抽象为：

Training：

Forward → Loss → Backward → Optimizer Step

Validation：

Forward → Validation Loss

只有 Training 分支会更新 Parameter。

In [14]:
torch.manual_seed(42)

model = nn.Linear(in_features=1, out_features=1)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

loss_fn = nn.MSELoss()
num_epochs = 100

for epoch in range(num_epochs):
    model.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad(set_to_none=True)

        prediction = model(x_batch)
        loss = loss_fn(prediction, y_batch)

        loss.backward()
        optimizer.step()

        batch_size = x_batch.shape[0]
        train_loss_sum += loss.item() * batch_size
        train_sample_count += batch_size

    train_loss = train_loss_sum / train_sample_count

    model.eval()

    valid_loss_sum = 0.0
    valid_sample_count = 0

    with torch.no_grad():
        for x_batch, y_batch in valid_loader:
            prediction = model(x_batch)
            loss = loss_fn(prediction, y_batch)

            batch_size = x_batch.shape[0]
            valid_loss_sum += loss.item() * batch_size
            valid_sample_count += batch_size

    valid_loss = valid_loss_sum / valid_sample_count

    if (epoch + 1) % 10 == 0:
        print(
            f"epoch={epoch + 1:03d}",
            f"train={train_loss:.6f}",
            f"valid={valid_loss:.6f}",
        )


epoch=010 train=0.237664 valid=0.515405
epoch=020 train=0.058739 valid=0.086087
epoch=030 train=0.045748 valid=0.030270
epoch=040 train=0.043565 valid=0.019312
epoch=050 train=0.043337 valid=0.017003
epoch=060 train=0.043220 valid=0.015752
epoch=070 train=0.043490 valid=0.017109
epoch=080 train=0.043235 valid=0.014778
epoch=090 train=0.043062 valid=0.016065
epoch=100 train=0.043639 valid=0.018812


## 17. Gradient Norm

Backward 完成后，每个 Parameter 都可能拥有 gradient。

全局 L2 Gradient Norm 常写为：

$$
\|g\|_2
=
\sqrt{
\sum_i
\|g_i\|_2^2
}
$$

Gradient Norm 是非常有用的训练诊断信号。

如果突然变得很大，可能意味着：

- Optimization 不稳定
- Learning Rate 太高
- Gradient Explosion
- 数值异常

如果长期接近 0，可能意味着：

- Gradient Vanishing
- 某些路径没有正常传播梯度

In [15]:
def global_gradient_norm(model: nn.Module) -> float:
    squared_norm_sum = 0.0

    for parameter in model.parameters():
        if parameter.grad is None:
            continue

        grad_norm = parameter.grad.norm(p=2)
        squared_norm_sum += grad_norm.item() ** 2

    return squared_norm_sum**0.5


## 18. Gradient Clipping

当 gradient norm 过大时，可以进行 Global Norm Clipping。

如果当前 gradient 为 \(g\)，最大允许 norm 为 \(c\)：

当：

$$
\|g\| \le c
$$

不做修改。

当：

$$
\|g\| > c
$$

进行缩放：

$$
g
\leftarrow
g
\frac{c}{\|g\|}
$$

Gradient Clipping 通常放在：

Backward 之后

Optimizer Step 之前

In [16]:
model.train()

x_batch, y_batch = next(iter(train_loader))

optimizer.zero_grad(set_to_none=True)

prediction = model(x_batch)

loss = loss_fn(prediction, y_batch)
loss.backward()

print("gradient norm before clipping:", global_gradient_norm(model))

max_grad_norm = 1.0

grad_norm_before_clip = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=max_grad_norm,
)

print("clip_grad_norm_ returned:", float(grad_norm_before_clip))
print("gradient norm after clipping:", global_gradient_norm(model))

optimizer.step()

gradient norm before clipping: 0.06152605043617228
clip_grad_norm_ returned: 0.06152604892849922
gradient norm after clipping: 0.06152605043617228


## 19. Training Metrics

训练时至少建议记录：

- Training Loss
- Validation Loss
- Learning Rate
- Gradient Norm
- Global Step

这些信息能帮助判断：

- Loss 是否正常下降
- 是否出现 Overfitting
- Learning Rate 是否合理
- Gradient 是否异常

In [17]:
def current_learning_rate(optimizer: torch.optim.Optimizer) -> float:
    return optimizer.param_groups[0]["lr"]


print("learning rate:", current_learning_rate(optimizer))

learning rate: 0.01


## 20. Checkpoint

训练中的 Checkpoint 不应只保存模型参数。

对于 AdamW 等带状态的 Optimizer，通常至少保存：

- `model.state_dict()`
- `optimizer.state_dict()`
- Epoch
- Global Step
- 其它需要恢复的训练状态

为什么 Optimizer State 也要保存？

因为 AdamW 内部维护：

- First Moment
- Second Moment
- Step Count

如果只恢复模型参数而重新创建 Optimizer，训练状态并不完整。

In [18]:
checkpoint_path = "lesson11_checkpoint.pt"

checkpoint = {
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "epoch": 100,
    "global_step": 1000,
}

torch.save(checkpoint, checkpoint_path)

print("saved:", checkpoint_path)

saved: lesson11_checkpoint.pt


In [19]:
new_model = nn.Linear(in_features=1, out_features=1)

new_optimizer = torch.optim.SGD(new_model.parameters(), lr=0.01)
checkpoint = torch.load(checkpoint_path, map_location="cpu")

new_model.load_state_dict(checkpoint["model"])
new_optimizer.load_state_dict(checkpoint["optimizer"])

print("epoch:", checkpoint["epoch"])
print("global_step:", checkpoint["global_step"])

epoch: 100
global_step: 1000


## 21. Language Modeling：Input / Target Shift

语言模型训练时，输入和目标通常来自同一个 token sequence，但错开一个位置。

原 sequence：

`t0, t1, t2, t3, t4`

Input：

`t0, t1, t2, t3`

Target：

`t1, t2, t3, t4`

也就是：

> 使用当前位置之前的 token 来预测下一个 token。

若 token batch shape 为：

$$
(B,T+1)
$$

则：

Input：

$$
(B,T)
$$

Target：

$$
(B,T)
$$

In [20]:
tokens = torch.tensor([[10, 11, 12, 13, 14], [20, 21, 22, 23, 24]])

inputs = tokens[:, :-1]
targets = tokens[:, 1:]

print("tokens :", tokens.shape)
print("inputs :", inputs.shape)
print("targets:", targets.shape)

print("inputs:")
print(inputs)

print("targets:")
print(targets)

tokens : torch.Size([2, 5])
inputs : torch.Size([2, 4])
targets: torch.Size([2, 4])
inputs:
tensor([[10, 11, 12, 13],
        [20, 21, 22, 23]])
targets:
tensor([[11, 12, 13, 14],
        [21, 22, 23, 24]])


## 22. Language Model Logits 与 Cross Entropy

Language Model 通常输出：

$$
logits.shape=(B,T,V)
$$

其中：

- \(B\)：Batch Size
- \(T\)：Sequence Length
- \(V\)：Vocabulary Size

Target：

$$
targets.shape=(B,T)
$$

为了使用常见的 Cross Entropy 接口，经常 reshape 为：

Logits：

$$
(BT,V)
$$

Target：

$$
(BT)
$$

也就是说，把所有 token position 看成一个更大的分类 batch。

In [21]:
B = 2
T = 4
V = 100

logits = torch.randn(B, T, V)
targets = torch.randint(low=0, high=V, size=(B, T))

flat_logits = logits.reshape(-1, V)
flat_targets = targets.reshape(-1)

print("logits      :", logits.shape)
print("targets     :", targets.shape)

print("flat_logits :", flat_logits.shape)
print("flat_targets:", flat_targets.shape)

logits      : torch.Size([2, 4, 100])
targets     : torch.Size([2, 4])
flat_logits : torch.Size([8, 100])
flat_targets: torch.Size([8])


In [22]:
loss_fn_lm = nn.CrossEntropyLoss()

loss = loss_fn_lm(flat_logits, flat_targets)

print("language model loss:", loss.item())

language model loss: 5.023869037628174


## 23. 最小 Language Model Training Step

现在可以把 Language Model 的单步训练抽象成：

Tokens

→ Input / Target Shift

→ Model

→ Logits

→ Cross Entropy

→ Backward

→ Gradient Clipping

→ Optimizer Step

真正进入 CS336 Assignment 1 后，模型会从这里逐渐替换为：

Embedding

→ Transformer Blocks

→ LM Head

## 24. 常见错误

### 错误 1：Validation 中忘记 `model.eval()`

某些 Layer 会继续按照 Training Mode 工作。

### 错误 2：认为 `model.eval()` 会关闭 gradient

不会。

Validation 中通常还需要 `torch.no_grad()`。

### 错误 3：Validation 中调用 `optimizer.step()`

Validation 不应该更新 Parameter。

### 错误 4：忘记清空 Gradient

PyTorch 默认会累积 gradient。

### 错误 5：Gradient Clipping 放错位置

正确顺序：

Backward → Gradient Clipping → Optimizer Step

### 错误 6：Checkpoint 只保存 Model，不保存 Optimizer

对于 AdamW 等 Optimizer，会丢失内部状态。

### 错误 7：LM Input 与 Target 没有错开一位

Language Modeling 的核心监督信号来自 Next-Token Prediction。